# Week 4: Probabilities, Bayes and naïve Bayes <span style="font-size: 0.3em;">v20260223d</span>

**Content:** 
- Part 1: Understanding the Bernoulli distribution
- Part 2: Understanding the univariate Normal distribution
- Part 3: Understanding the multivariate Normal distribution
- Part 4: Bayes and naive Bayes classification
- [Assignment 4: Whisky dataset (marginals, conditionals, Bayes, Naïve Bayes)](#my-anchor)

**Objective:**
- Understand the standard probability distributions and use them in machine learning contexts.

## Commands and Methods Used in the Notebook

The exercises use (or ask you to use) the following commands and methods:

- `numpy`
  - `np.array`, `np.asarray`: create or convert to NumPy array
  - `np.random.seed`, `np.random.rand`, `np.random.binomial`, `np.random.normal`, `np.random.uniform`, `np.random.randn`, `np.random.multivariate_normal`: random sampling
  - `np.mean`, `np.median`, `np.std`, `np.sum`, `np.nansum`: statistics
  - `np.linspace`, `np.zeros`, `np.concatenate`: array construction
  - `np.linalg.norm`: vector/matrix norm
  - `np.isclose`, `np.isin`: comparisons
  - `np.where`, `np.histogram2d`: indexing and histograms
  - `np.loadtxt`: load text files
- `scipy`
  - `scipy.stats`: e.g. `stats.norm.pdf` for Normal PDF
  - `scipy.spatial.distance.pdist`, `squareform`: pairwise distances
  - `scipy.special.logsumexp`: log-sum-exp for numerical stability
- `sklearn`
  - `sklearn.naive_bayes.MultinomialNB`, `BernoulliNB`: Naïve Bayes classifiers
  - `sklearn.preprocessing.OneHotEncoder`: encode categorical features
  - `.fit`, `.predict`, `.fit_transform`: model fitting and prediction
- `pandas`
  - `pd.read_csv`: load CSV data
  - `DataFrame.drop`, `.values`: drop columns, get NumPy array
- `matplotlib.pyplot`: We use various plotting functions

**Run the cell below first.** It loads all libraries and settings used throughout this notebook.

In [ ]:
# Imports and settings used throughout the notebook
import re
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.special import logsumexp

from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.preprocessing import OneHotEncoder

import matplotlib.pyplot as plt
import seaborn as sns

# Plotting style
sns.set_style('darkgrid')
sns.set_theme(font_scale=1.)


## Introduction - recap on basic probability theory in Python

In the first part of today's exercise, we review core concepts of probability theory and apply them using NumPy for simulation. We will have a closer look at joint, conditional probability and marginal probability, the law of total probability, independence and Bayes' theorem. The goal is to connect probability formulas to reproducible Python simulations, that we can use for modeling later on.

We consider a working example of students who just finished a machine learning (ML) course and let the events
- $A$ = student attended the lectures and exercises sessions during the course
- $P$ = student passed the final exam

#### Joint and conditional probabilities

The joint probability of two events $A$ and $P$ measures the likelihood that both events occur simultaneously, i.e.
$$
    P(A \cap P) = \text{probability that a student both attended and passed the exam.}
$$
Conditional probability measures the likelihood of one event given that the other has occurred. It is directly linked to the joint probability by division with the probability of one of the events, e.g.
$$
    P(A \mid P) = \frac{P(A \cap P)}{P(P)} \quad \Leftrightarrow \quad P(A \cap P) = P(A \mid P) P(P)
$$

After the exame, we asked students who passed and did not pass if they attended the course sessions. Say we got the following statistics:
- $P(P) = 0.75$
- $P(A \mid P) = 0.99$
- $P(A \mid \neg P) = 0.1$

**Task I.1:** Compute the joint probabilities $P(A, P)$ and $P(A, \neg P)$ using the formula. Additionally, the code simulates a class of $N = 600$ students and checks that you get the correct quantities.

**Step-by-step:** (1) Use $P(A \cap P) = P(A \mid P)\, P(P)$ and $P(A \cap \neg P) = P(A \mid \neg P)\, P(\neg P)$ to compute the two joint probabilities. (2) For the simulation: first draw who passed, $P$, with $P(P)=0.75$; then for those who passed, draw attendance $A$ with probability $P(A\mid P)$; for those who did not pass, draw $A$ with probability $P(A\mid \neg P)$. (3) From the simulated data, estimate $P(A,P)$ as the fraction of students who both attended and passed, and $P(A,\neg P)$ as the fraction who attended but did not pass.

> *Hint:* Use `np.random.rand(N) < p` to generate binary outcomes. Use logical conditions like `&`, `|` and `~` to combine events $P$ and $A$.

> *Hint:* Remember the complement rule: $P(\neg P) = 1 - P(P)$.

In [ ]:
# Number of students
N = 600


np.random.seed(42)  # for reproducibility (Task I.3)

# Define measures
P_P = 0.75
P_A_given_P = 0.95
P_A_given_not_P = 0.1

# Compute joint probabilities and name the variables joint_A_P and joint_A_not_P
# YOUR CODE HERE
raise NotImplementedError()

print(f"P(P): {P_P:.2f}")
print(f"P(A|P): {P_A_given_P:.2f}")
print(f"P(A|~P): {P_A_given_not_P:.2f}")
print(f"Joint P(A, P): {joint_A_P:.2f}")
print(f"Joint P(A, ~P): {joint_A_not_P:.2f}")

# Simulate: first draw who passed (P), then attendance (A) given P
P_sim = np.random.rand(N) < P_P
n_passed = P_sim.sum()
n_failed = N - n_passed
A_given_P_sim = np.random.rand(n_passed) < P_A_given_P
A_given_not_P_sim = np.random.rand(n_failed) < P_A_given_not_P
A_sim = np.zeros(N, dtype=bool)
A_sim[P_sim] = A_given_P_sim
A_sim[~P_sim] = A_given_not_P_sim

print(f"\nSimulated P(P): {P_sim.mean():.2f}")
print(f"Simulated P(A|P): {A_given_P_sim.mean():.2f}")
print(f"Simulated P(A|~P): {A_given_not_P_sim.mean():.2f}")
print(f"Simulated P(A,P): {(A_sim & P_sim).mean():.2f}")
print(f"Simulated P(A,~P): {(A_sim & ~P_sim).mean():.2f}")


As you see, re-running your code will give a different estimate of the empirical probabilities thus resembling variation in new semesters of students taking the course. When doing simulation-based experiments this is a bad property as it means that we are unable to reproduce the exact results seen once. Luckily, we can fix the pseudo-randomness in Python, thereby ensure reproducibility.

**Task I.2:** For some reason, the popularity of the course changed dramatically from one to another semester resulting in a total of $N=10{,}000$ registered course participants. Say our survey statistics about passing the exam and attending the course remain unchanged, how will this increase in students affect your simulation estimates?

- *Answer:* With more students, the simulated proportions get closer to the true probabilities (less random variation).

**Task I.3:** Use `np.random.seed(42)` (or another number) at the start of your simulation so that the random draws are fixed. Re-run the cell several times and check that the estimated probabilities no longer change.

#### Law of Total Probability

In many scenarios, we are not able of directly measure the marginal probability of an event, e.g. $P(A)$. Measuring student attendence can be difficult, yet we do know the conditional probabilities of attending the course given pass/not pass from the follow-up survey. The law of total probability allows us to compute the marginal probability $P(A)$ by summing over mutually exclusive events $P_i$:
$$
    P(A) = \sum_i P(A \mid P_i) \, P(P_i) = P(A \mid P) \, P(P) + P(A \mid \neg P) \, P(\neg P) 
$$

**Task I.4:** Compute the marginal probability of attending the course, $P(A)$, using the law of total probability: $P(A) = P(A\mid P)P(P) + P(A\mid \neg P)P(\neg P)$. Use the same numbers as above ($P(P)$, $P(A\mid P)$, $P(A\mid \neg P)$).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
print(f"Probability of attending the course: {P_A:.2f}")

#### Independence

The two events $A$ and $P$ are independent if knowing that one occurs does not change the probability of the other, i.e.
$$
    P(A \cap P) = P(A) \ P(P)
$$
Otherwise, the events are dependent.

**Task I.5:** Check if course attendance and passing the exam are independent using your previous calculation. Explain your result.

- *Answer:* They are not independent: $P(A \cap P) \neq P(A)\, P(P)$. So knowing whether someone attended changes the probability that they passed.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

#### Bayes' theorem

We now have all the building blocks for establishing Bayes’ theorem that allows us to reverse conditional probabilities, computing the probability of a cause given an observed effect, i.e.
$$
    P(P \mid A) = \frac{P(A \cap P)}{P(A)} = \frac{P(A \mid P)P(P)}{P(A)} = \frac{P(A \mid P) \ P(P)}{P(A \mid P) \, P(P) + P(A \mid \neg P) \, P(\neg P) }
$$
As you can see, Bayes' theorem uses the relation between the joint and conditional probability, the law of large numbers and the complement rule. Similarly we can use Bayes' theorem to compute the probability of a cause given not observing an effect, i.e.
$$
    P(P \mid \neg A) = \frac{P(\neg A \cap P)}{P(\neg A)} = \frac{P(\neg A \mid P)P(P)}{P(\neg A)} = \frac{(1 - P(A \mid P)) P(P)}{1 - P(A)} = \frac{(1 - P(A \mid P)) P(P)}{1 - P(A \mid P) \, P(P) + P(A \mid \neg P) \, P(\neg P) }
$$

**Task I.6:** Estimate the probabilities that:
1) a student passed given that they attended the course sessions. (given)
2) a student passed given that they did not attend the course sessions. (given)
3) a student did not pass given that they attended the course sessions. (to be computed in the excercise)
4) a student did not pass given that they did not attend the course sessions. (to be computed in the excercise)

Based on your results, do you think it is generally a good idea to attend the course sessions?

> *Hint:* Again, remember the complement rule. It also works as $P(\neg P \mid A) = 1 - P(P \mid A)$.

In [ ]:


# Use Bayes theorem with the previously computed probabilities
P_P_given_A = joint_A_P / P_A
P_not_P_given_A = 1 - P_P_given_A

#Name the variables P_P_given_not_A and P_not_P_given_not_A
# YOUR CODE HERE
raise NotImplementedError()

print(f"Probability of passing given attending: {P_P_given_A:.2f}")
print(f"Probability of not passing given attending: {P_not_P_given_A:.2f}")
print(f"\nProbability of passing given not attending: {P_P_given_not_A:.2f}")
print(f"Probability of not passing given not attending: {P_not_P_given_not_A:.2f}")



---

## Part 1: Understanding the Bernoulli distribution

The Bernoulli distribution models a random experiment with exactly two possible discrete outcomes - "success" (coded as 1) and "failure" (coded as 0) - and its' probability mass function (PMF) is
$$
    p(b|\theta) = \theta^b (1 - \theta)^{1-b}
$$
where $\theta \in [0,1]$ is the success rate. The Bernoulli distribution satisfies $p(b=1|\theta) = \theta$ and $p(b=0|\theta) = 1 - \theta$ and we can estimate the empirical success rate from $N$ samples, $\left[b_1, b_2, \dots b_N\right]$ as
$$
    \hat{p}\left({b=1}\right) = \frac{1}{N} \sum_{i=1}^N \mathbb{I}[b_i = 1]
$$
where $\mathbb{I}[\cdot]$ is the indicator function. If we run multiple independent Bernoulli experiments in a sequence, the resulting PMF distributes according to the [Binomial distribution](https://en.wikipedia.org/wiki/Binomial_distribution).

**Task 1.1:** Below observe that $N=200$ samples from a Bernoulli distribution with success rate $\theta=0.8$ are drawn and plotted (e.g. as a sequence of 0s and 1s or a bar chart).
> *Hint:* Use `np.random.binomial(n=1, p=0.8, size=N)` to get one trial per sample (so each outcome is 0 or 1).

**Task 1.2:** From the same samples, compute the empirical PMF: $\hat{p}(b=0)$ = proportion of zeros, $\hat{p}(b=1)$ = proportion of ones. These two probabilities are plotted as a bar plot with `plt.bar()` (e.g. x = [0, 1], height = [proportion of 0s, proportion of 1s]).

In [ ]:

# Set parameters for Bernoulli distribution
N = 200
p = 0.8

# Generate Bernoulli samples
X = np.random.binomial(1, p, N)

# Plot the samples
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(r"Bernoulli distribution ($\theta=0.8)$")

axs[0].plot(X, ".", alpha=0.7)
axs[0].set_title("Bernoulli samples")
axs[0].set_xlabel("Sample index")
axs[0].set_ylabel("Value")

# Compute empirical probabilities and name the variables p0 and p1
# YOUR CODE HERE
raise NotImplementedError()

# Bar plot for empirical PMF
axs[1].set_title("Probability mass function (empirical)")
axs[1].bar([0, 1], [p0, p1], color=['blue', 'orange'], alpha=0.7)
axs[1].set_xticks([0, 1])
axs[1].set_ylabel(r"$p(b|theta)$")
axs[1].set_xlabel(r"$b$")
plt.show()

print(f"Empirical probability of success (b=1): {p1:.3f}")
print(rf"Theoretical probability of success (theta): {p:.3f}")



**Task 1.3:** Argue why the empirical success rate $\hat{p}\left(b=1\right)$ is close but not equal to the theoretical succes rate $p\left(b=1|\theta\right)=\theta$ used to generate the samples.
- *Answer:* 

<br>

---

## Part 2: Understanding the univariate Normal distribution

The Normal distribution - also known as the Gaussian distribution - is central to many methods in statistics and machine learning, both in its univariate and multivariate forms. It is a continuous probability distribution characterized by its bell-shaped curve and is fully defined by its mean, $\mu$, and variance, $\sigma^2$, which determine its center and spread. 

If a real-valued random variable $x\in\mathbb{R}$ is Normally distributed, its' probability density function (PDF) follows
$$
    \mathcal{N}\left(x|\mu, \sigma^2\right) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(- \frac{x - \mu^2}{2\sigma^2}\right)
$$
the empirical mean and variance of $N$ samples $x_1, x_2, \dots, x_N$ are given by:
$$
    \hat{\mu} = \frac{1}{N} \sum_{i=1}^N x_i \qquad \text{and} \qquad \hat{\sigma}^2 = \frac{1}{N-1} \sum_{i=1}^N (x_i - \hat{\mu})^2
$$

Here, $\hat{\mu}$ provides an estimate of the true mean $\mu$ of the underlying distribution, while $\hat{\sigma}^2$ is the empirical variance (using $N-1$ in the denominator for an unbiased estimate). These statistics summarize the central tendency and spread of the observed data. In practice, the empirical mean and variance will vary dependent on the drawn samples, but as $N$ increases, they converge to the true parameters of the distribution due to the [Law of Large Numbers](https://en.wikipedia.org/wiki/Law_of_large_numbers).


**Task 2.1:** Generate $N=200$ samples from a Normal distribution with mean $\mu = 17$ and standard deviation $\sigma = 2$. Plot the samples (e.g. as a 1D scatter or line) and a histogram of the samples.
> *Hint:* Use `np.random.normal(loc=mu, scale=sigma, size=N)` to generate the samples. Use `plt.plot()` or a 1D scatter and `plt.hist()` for the histogram.

In [ ]:
# Define parameters
N = 200
mu = 17
sigma = 2

# YOUR CODE HERE
raise NotImplementedError()

**Task 2.2:** The histogram is generated by counting how many of the samples fall within the range covered by each bin of the histogram. Try changing the number of bins in the histogram.

**Task 2.3:** Compute the empirical mean and standard deviation of the generated samples. Show, that they are close but not equal to the theoretical values used to generate the random
samples.
> *Hint:* Take a look at the functions `np.mean()` and `np.std()`

**Task 2.4:** Try running the code a few times and see how the empirical mean and standard deviation changes when new random numbers are generated from the same distribution.
> *Hint:* Remember to re-run the previous cell where the data is generated !

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print("Theoretical mean: ", mu)
print("Theoretical std.dev.: ", sigma)
print("Empirical mean: ", mu_)
print("Empirical std.dev.: ", sigma_)

**Task 2.5:** The empirical mean and theoretical mean are plotted, plot the true probability density function (PDF) on top of your histogram.
> *Hint:* Take a look at the function `scipy.stats.norm.pdf()` and `plt.axvline()`

In [ ]:
# Plot the histogram
fig = plt.figure()
plt.title("Normal distribution")
plt.hist(X, bins=nbins, density=True)

# Over the histogram, plot the theoretical probability distribution function:
# YOUR CODE HERE
raise NotImplementedError()

# Plot the empirical mean and theoretical mean
plt.axvline(mu_, color='r', linestyle='--', label='Empirical mean')
plt.axvline(mu, color='g', linestyle='--', label='Theoretical mean')
plt.legend()
plt.show()



**Task 2.6:** Show that when the number of samples $N$ is increased, the histogram approximates the pdf better and the empirical estimates of the mean and standard deviation improve.


#### The Central Limit Theorem
One reason that the Normal distribution is central in machine learning is due to the **Central Limit Theorem (CLT)**. The CLT states that the sample mean $\bar{x} = \frac{1}{N} \sum_i^N x_i$ of a collection of i.i.d. random variables, $\mathcal{S}=\{x_i\}_{i=1}^N$, follows a Normal distribution for large sample size $N$. This holds for any data-generating distribution, i.e. $x_i \sim p\left(\mu,\sigma^2\right)$, with mean $\mu$ and finite variance $\sigma^2$. The sample mean is indeed a random variable itself, as it depends on the specific realization of $\mathcal{S}$. Specifically, the CLT states that
$$
\bar{x} \sim \mathcal{N}\left(\mu, \frac{\sigma^2}{N}\right)
$$
for large $N$. This allows us to easily make statistical inference such as providing uncertainty estimates even if the original data distribution is nowhere near being Gaussian. In the following exercise, we will give a simulation-based "proof" that this is indeed true using the continuous uniform distribution for generating the sample collection, i.e. $x_i \sim \operatorname{Uniform}(0,1)$. 

**Task 2.7:** Observe in the code how to generate $N=50$ samples from $\operatorname{Uniform}(a,b)$ and compute the sample mean, $\bar{x}$. The samples are plotted as a histogram.
> *Hint:* Use `np.random.rand()` or `np.random.uniform()`.

**Task 2.8:** Repeat this computation yourself to get $S=1000$ realizations of the sample mean. Plot this empirical sample mean distribution as a histogram. Plot the PDF of the Normal distribution with mean $\mu$ and variance $\frac{\sigma^2}{N}$ on top.
> *Hint:* The $\operatorname{Uniform}(a,b)$ distribution has mean $\mu=\frac{1}{2} (a + b)$ and variance $\sigma^2 = \frac{1}{12} (b-a)^2$. For other distributions, [see here](https://mathcs.clarku.edu/~djoyce/ma217/distributions2.pdf).

In [ ]:

N = 50
S = 1000
a = 0
b = 1

# Sample N samples from a uniform distribution, S times
X_s= np.random.uniform(a, b, (S, N))

# Plot the histogram of samples from one realization 
plt.figure(figsize=(8, 4))
plt.hist(X_s[0], bins=10, density=True, alpha=0.6, color='blue')
plt.title("Histogram of Uniform Distribution Samples")
plt.xlabel("Value")
plt.ylabel("Density")
plt.grid(True)
plt.show()

# YOUR CODE HERE
raise NotImplementedError()

**Task 2.9:** Try to decrease the number of samples $N$. How does this affects the sample mean distribution? What happens if you modify the range of the uniform distribution?

**Optional:** Modify the data-generating distribution to any distribution you can think of. Does this choice impact the number of samples requires for the CLT to accurately describe the sample mean distribution? 

<br>

---

## Part 3: Understanding the multivariate Normal distribution

So far we have been considered a 1-dimensional Normal distributed random variable but we will now proceed to the *multivariate Normal distribution*. The probability density function (PDF) of the $M$-dimensional multivariate Normal distribution is
$$
    \mathcal{N}\left(\boldsymbol{x}|\boldsymbol{\mu}, \Sigma\right) = \frac{1}{\sqrt{\left(2\pi\right)^M \lvert \bf{\Sigma} \rvert}} \exp\left(-\frac{1}{2} \underbrace{\left(\boldsymbol{x} - \boldsymbol{\mu}\right)^\top \bf{\Sigma}^{-1} \left(\boldsymbol{x} - \boldsymbol{\mu}\right)}_{d} \right)
$$
where $\boldsymbol{\mu}=\left[\mu_1, \mu_2 \right]^\top$ is a vector containing the mean values for each dimension and $\bf{\Sigma}$ is the covariance matrix. Taking a closer look at the expression, we see that the leading term is $d=\left(\boldsymbol{x} - \boldsymbol{\mu}\right)^\top \bf{\Sigma}^{-1} \left(\boldsymbol{x} - \boldsymbol{\mu}\right)$ which is exactly the *Mahalanobis distance* that gives distances between a sample realization $\boldsymbol{x}$ and the mean $\boldsymbol{\mu}$ while respecting the covariance structure of the distribution through $\bf{\Sigma}^{-1}$. The bell-shape arises from applying the exponential to this distance. Note that the distribution with $\boldsymbol{\mu}=\bf{0}$ and $\bf{\Sigma} = I$ is called the *standard multivariate Normal distribution* for all dimensionalities $M$.

In the next step we will consider the multivariate Normal distribution in two dimensions. The covariance matrix for a 2D Gaussian is described by
$$
\bf{\Sigma } = \begin{bmatrix}
      \sigma _1^2 & \mathop{\rm cov} \left( x_1, x_2 \right)  \\
      \mathop{\rm cov} \left( x_2, x_1 \right) & \sigma _2^2
\end{bmatrix}
$$

where $\mathrm{cov}(\cdot,\cdot)$ is covariance between two random variables. Note that ${\mathop{\rm cov}} \left( { {x_1},{x_2}} \right) = {\mathop{\rm cov}}\left( { {x_2},{x_1}} \right)$, i.e., the covariance matrix is symmetric, and $\sigma_n^2={\mathop{\rm cov}}\left( { {x_n},{x_n}} \right)$. We can write the covariance matrix in terms of the correlation between attributes as
$$
      \mathop{\rm Correlation}\left( { {x_1},{x_2}} \right) = \frac{ {\mathop{\rm cov}} \left( { {x_1},{x_2}} \right)}{\sqrt{ {\mathop{\rm cov}} \left( { {x_1},{x_1}} \right)}\sqrt{ {\mathop{\rm cov}} \left( { {x_2},{x_2}} \right)}} \qquad \Leftrightarrow \qquad {\mathop{\rm cov}}\left( { {x_1},{x_2}} \right) = {\mathop{\rm Correlation}} \left( { {x_1},{x_2}} \right)\sqrt{ {\mathop{\rm cov}} \left( { {x_1},{x_1}} \right)}\sqrt{ {\mathop{\rm cov}} \left( { {x_2},{x_2}}\right) }.
$$

**Task 3.1:** Generate $N=1000$ samples from a 2-dimensional Normal distribution with mean $\boldsymbol{\mu}=\begin{bmatrix} 13 & 17 \end{bmatrix}$, $\sigma_1=2$, $\sigma_2=3$ and $\mathop{\rm{Correlation}}(x_1,x_2)=0.5$. Your task is to define the covariance matrix so that the code below can generate the samples
> *Hint:* Define the covariance matrix $\bf{\Sigma}$ from the given information.

> *Hint:* Look at the function `np.random.multivariate_normal()` to learn how you can generate multivariate Normal distributed random numbers in Python.

In [ ]:

# Define parameters
N = 1000
sigma1 = 2
sigma2 = 3
corr = 0.5

# Define mean vector and covariance matrix and name them mu and Sigma
mu = np.array([13, 17])

# YOUR CODE HERE
raise NotImplementedError()
# Generate samples from the Normal distribution
X = np.random.multivariate_normal(mu, Sigma, N)


**Task 3.2:** Plot the generated samples as a scatter plot as well as a 2-dimensional histogram (code already does this). Choose a suitable number of histogram bins by visual inspection.
> *Hint:* Use `np.histogram2d()` or `plt.hist2d()` to create a 2-dimensional histogram.

In [ ]:
# Number of bins in histogram and name it nbins
# YOUR CODE HERE
raise NotImplementedError()
# Plot scatter plot of data
fig, axs = plt.subplots(1, 2, figsize=(10,6), sharex=True, sharey=True)
fig.suptitle("2-D Normal distribution")

axs[0].set_title("Scatter plot of data")
axs[0].plot(X[:, 0], X[:, 1], "x")
axs[0].set_xlabel("x1")
axs[0].set_ylabel("x2")

axs[1].set_title("2D histogram")
axs[1].hist2d(X[:, 0], X[:, 1], bins=nbins, cmap='Blues')
x = np.histogram2d(X[:, 0], X[:, 1], nbins)
hist_img = axs[1].imshow(x[0], cmap='Blues', interpolation="None", origin="lower")
fig.colorbar(hist_img, ax=axs[1], label='Counts', shrink=0.8)
axs[1].set_xlabel("x1")
plt.tight_layout()
plt.show()


**Task 3.3:** Show that when the correlation between $x_1$ and $x_2$ is zero, the scatter plot and 2-d histogram have the shape of an axis-aligned ellipse. Can you explain why?
- *Answer:* 

**Task 3.4:** Show that when the correlation between $x_1$ and $x_2$ is one, the values of $x_1$ and $x_2$ fall on a straight line. Can you explain why?
- *Answer:*

**Task 3.5:** Try varying the number of samples, the mean, the standard deviations, the correlation and the number of histogram bins and see what happens.

#### The Curse of Dimensionality

The **curse of dimensionality** describes a collection of phenomena that occur when working with data in high-dimensional spaces. As the number of dimensions, $M$, increases, the volume of the space grows exponentially, making data points increasingly far apart relative to the space - this means that the data becomes *sparse*. This sparsity has several important consequences:

- **Distance becomes less meaningful:** In high dimensions, the minimum and maximum pairwise distances between points become very similar, meaning that all points tend to be nearly equidistant from each other. This can cause problems for machine learning methods such as $k$-nearest neighbors and clustering techniques that rely on distance metrics.
- **Increased data and computation requirements:** To cover a high-dimensional space adequately, exponentially more data is needed. This raises both computational costs and the risk of overfitting, especially when training complex models on limited data (a topic we will explore further later on).

To explain this more precisely, consider a collection of $N$ data points $\{\boldsymbol{x}_i = \left[x_{i,1}, x_{i,2}, \dots, x_{i, M}\right]^\top\}_{i=1}^N$ drawn from an $M$-dimensional standard multivariate normal distribution, $\boldsymbol{x}_i \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$. 
The squared Euclidean norm of a single samples is given by $\|\boldsymbol{x}_i\|^2 = \sum_{j=1}^M x_{ij}^2$ which follows a $\chi^2(M)$ distribution. This distribution has mean $M$ and variance $2M$, so by the law of large numbers - stating that the sample mean converges to the true mean for large $N$ - the squared norm $\|\boldsymbol{x}_i\|^2$ concentrates around its expected value $M$. Consequently, the Euclidean norm $\|\boldsymbol{x}_i\|$ that represents the distance to origo, grows by $\sqrt{M}$ when increasing the dimensionality $M$. 

We will now explore a simple example to demonstrate these effects and discuss their implications for machine learning models.

**Task 3.6:** Generate $N=100$ samples from standard multivariate Normal distributions with dimensionalities $M \in [1, 10, 100, 1000]$. Compute the squared Euclidean distances to origo and summarize these through the mean and standard deviation for each dimensionality $M$.
> *Hint:* Store the results in variables `mean_distances` and `std_distances`. We use `plt.errorbar()` for plotting these results.

> *Hint:* Use `np.linalg.norm()` for computing the squared Euclidean distance.

> *Hint:* Verify that your results align with the theoretical expectation $\sqrt{M}$.

In [ ]:
num_samples = 100
dims = [1, 10, 100, 1000]

distances = np.zeros((len(dims), num_samples))  # sets up storage for distances
for i, M in enumerate(dims):
    X = np.random.randn(num_samples, M)
    distances[i] = np.linalg.norm(X, axis=1)    # the squared Euclidean distance to origin

# Calculate the mean distance and standard deviation for each dimensionality and name them mean_distances and std_distances
# YOUR CODE HERE
raise NotImplementedError()

fig = plt.figure(figsize=(6, 4))
plt.title('Mean distance from origin vs. dimensionality')
plt.errorbar(dims, mean_distances, yerr=std_distances, fmt='-x', capsize=5)
plt.plot(np.logspace(0, 3, 100), np.sqrt(np.logspace(0, 3, 100)), 'r--', label=r'Expected: $\sqrt{M}$')
plt.xlabel('Dimensionality')
plt.ylabel('Mean distance from origin')
plt.xscale('log')
plt.legend()
plt.show()

**Task 3.7:** Generate $N=100$ samples from standard multivariate Normal distributions with dimensionalities $M \in [1, 10, 100]$. For each dimensionality, compute the pairwise Euclidean distance matrix, $\bf{D}$, between samples and plot the results.
> *Hint:* The pairwise distance matrix can be computed with `scipy.spatial.distance.pdist()` and `scipy.spatial.distance.squareform()`.

> *Hint:* You can plot matrices using `plt.imshow()`.

In [ ]:
num_samples = 100
dims = [1, 10, 100]


fig, axs = plt.subplots(1, len(dims), figsize=(12, 4))
fig.suptitle('Pairwise distances in different dimensions')

# Loop through each dimensionality and compute pairwise distances
# YOUR CODE HERE
raise NotImplementedError()

**Task 3.8:** Based on the above experiments, what problems does the "curse of dimensionality" introduce in the context of the machine learning methods discussed in the course so far?
> *Hint:* What happens to the distance between points when you increase the dimensionality?

> *Hint:* Can you explain why, in high dimensions, the minimum and maximum pairwise distances between points get closer together?

> *Hint:* Why is this problematic for distance-based methods like $k$-nearest neighbors?

- *Answer:* 

**Task 3.9 (optional):** Try to read-through or re-implement [this exercise](https://www.geeksforgeeks.org/machine-learning/k-nearest-neighbors-and-curse-of-dimensionality/). The idea is to fit a $k$-nearest neighbor classifier on two data sets: 1) a high-dimensional image data and 2) a lower-dimensional version constructed using PCA. Which classifier performs best and why?

<br>

---

## Part 4: Bayes and Naive Bayes

We will now use our understanding of the previous distributions to examine two classifiers, namely a Bayesian classifier and a Naive Bayes classifier. Recall that the Bayesian classifier follows directly from Bayes' theorem:
$$
    p\left(y=c | \boldsymbol{x}\right) = \frac{p\left(\boldsymbol{x} | y=c\right) p(y=c)}{\sum_{k=0}^{K-1} p\left(\boldsymbol{x} | y=k\right) p(y=k)}
$$
where $K$ is the number of classes, $p\left(\boldsymbol{x}|y\right) = p\left(x_1, x_2, \dots, x_M | y\right)$ is the likelihood and $p(y)$ is the prior class information. The only difference between this Bayesian classifier and the Naive Bayes classifier, is the Naive Bayes assumption, stating independence of the likelihood terms given the label, i.e. $p\left(x_1, x_2, \dots, x_M | y\right) = p\left(x_1|y\right) p\left(x_2|y\right) \times \dots \times p\left(x_M|y\right)$. This results in a classifier of the following form:
$$
    p\left(y=c | \boldsymbol{x}\right) = \frac{p\left(x_1|y=c\right) p\left(x_2|y=c\right) \times \dots \times p\left(x_M|y=c\right) p(y=c)}{\sum_{k=0}^{K-1} p\left(x_1|y=k\right) p\left(x_2|y=k\right) \times \dots \times p\left(x_M|y=k\right) p(y=k)}
$$

In the next part of the exercise we will classify names as female or male names - for further details see also the `readme_male_female.txt` file in the associated data folder. 

We have a database with 2943 male names and 5001 female names. We will only consider names that contain at least four letters resulting in a total of $N^{\text{Male}} = 2785$ male and $N^{\text{Female}} = 4866$ female names.

**Task 4.1:** Examine and load the data files provided in the associated data folder. Filter the data to only consider names that contain at least four letters.
> *Hint:* Use `np.loadtxt()` to load txt-files. Be sure to set the `delimiter` correctly.

In [ ]:
# Load male names from file
male_names = np.loadtxt('data/names/male.txt', dtype=str, delimiter='\\')
male_names = male_names[[len(name) >= 4 for name in male_names]]
# Or alternatively:
# male_names = male_names[np.char.str_len(male_names) >= 4]

# YOUR CODE HERE
raise NotImplementedError()
# Concatenate in a single array
names = np.concatenate((male_names, female_names))


assert len(male_names) == 2785, "There should be 2785 male names!"
assert len(female_names) == 4866, "There should be 4866 female names!"

As feature for the classification we will use the first and second letter as well as the second last and last letter of the names denoted respectively $x_1$, $x_2$, $x_3$ and $x_4$. As an example, the name "Richard" will have $x_1=r$, $x_2=i$, $x_3=r$, $x_4=d$. Therefore, $x_i\in\{a,b,c,d,\ldots ,z\}$ such that each feature can take the value of any of the 26 letters of the alphabet (from a to z) hence the attributes used are discrete/categorical. In Python, we code each letter as a numbers between 1 and 26 where 1 corresponds to a and 26 to z.

**Task 4.2:** Inspect the following code that generates the data matrix $\boldsymbol{X}$ and binary target vector $\boldsymbol{y}$ with $y=0$ corresponding to male names and $y=1$ corresponding to female names.

In [ ]:
# We construct a function that converts a name to a vector of letter indices.
# The vector will contain the indices of the first two and last two letters of the name.
def name2vector(name):
    """Convert a name to a vector of letter indices."""
    assert len(name) >= 4, "Name must have at least 4 characters!"
    name = name.strip().lower()
    return np.array([
        ord(name[0]) - ord("a") + 1,
        ord(name[1]) - ord("a") + 1,
        ord(name[-2]) - ord("a") + 1,
        ord(name[-1]) - ord("a") + 1
    ], dtype=int)

# Create the structure for the data matrix
X = np.zeros((len(names), 4), dtype=int)
# Create the target vector
y = np.concatenate([[0] * len(male_names) + [1] * len(female_names)]).reshape(-1,1)

# Loop through each name in the dataset
for i, name in enumerate(names):
    # Convert the name to a vector of letter indices
    letter_idxs = name2vector(name)
    # Fill the data matrix with letter indices
    X[i, :] = letter_idxs

assert X.shape == (7651, 4), "X should have shape (7651, 4)!"
assert y.shape == (7651, 1), "y should have shape (7651, 1)!"
assert np.all(np.isin(X, np.arange(1, 27))), "X should only contain values from 1 to 26!"

Using the formulation of a Bayesian classifier, we can classify a name as a female name if $p(y=1|x_1,x_2,x_3,x_4) > p(y=0|x_1,x_2,x_3,x_4)$ and as a male name otherwise. In order to classify names as male or female we need to compute our likelihood which is the number of times a given letter combination occurred for the male and female names, respectively. In other words we need to count how many times each of the letter combinations 
$$
\begin{align}
    (x_1,x_2,x_3,x_4) &=(a,a,a,a) \\ 
    (x_1,x_2,x_3,x_4) &=(a,a,a,b) \\ 
    &\vdots \\
    (x_1,x_2,x_3,x_4) & =(z,z,z,z)
\end{align}
$$ 
occurred in each of the two classes to get $p(x_1, x_2, x_3, x_4|y=0)$ and $p(x_1, x_2, x_3, x_4|y=1)$.

**Task 4.3:** How many different letter combinations do we have to evaluate? How well do you think we can identify the probabilities $p(x_1, x_2, x_3, x_4|y=0)$ and $p(x_1, x_2, x_3, x_4|y=1)$ from the data at hand? 
> *Hint:* Remember your combinatorics - you have to use the [Rule of Product](https://en.wikipedia.org/wiki/Rule_of_product).

> *Hint:* Is it likely that a given dataset will contain enough samples to allow us to accurately estimate these probabilities?

- *Answer:* 

Rather than finding the full joint distribution $p(\boldsymbol{x}|y)$ we will use the naive Bayes assumption. Using Naive Bayes we only need to estimate the probability of observing each of the 26 letters for each of the four considered position in the spelling of the name separately, i.e.
$$
\begin{align}
    &p(x_1 = a| y), \quad p(x_1 = b| y), \quad \dots, \quad p(x_1 = z| y) \\
    &p(x_2 = a| y), \quad p(x_2 = b| y), \quad \dots, \quad p(x_2 = z| y) \\
    &p(x_3 = a| y), \quad p(x_3 = b| y), \quad \dots, \quad p(x_3 = z| y) \\
    &p(x_4 = a| y), \quad p(x_4 = b| y), \quad \dots, \quad p(x_4 = z| y)
\end{align}
$$
as well as the prior class probabilities $p(y=0)$ and $p(y=1)$. We can empirically estimate these likelihood terms by counting the fraction of times a letter, $x_t$, occured at the $i$'th position in male and female names, respecitve class. That is:
$$
    p(x_i = x_t | y=0) = \frac{N^{\text{Male}}(x_i = x_t)}{N^{\text{Male}}} \qquad \text{and} \qquad p(x_i = x_t | y=1) = \frac{N^{\text{Female}}(x_i = x_t)}{N^{\text{Female}}}
$$
From more than one thousand male and female names we can quite accurately estimate these probabilities.

**Task 4.4:** Fit a Naive Bayes classification model to the data with a uniform prior. Compute the posterior probabilities and use them to classify the names. Compute the misclassification rate and print the first 100 misclassified names - why do you think these names were misclassified?
> *Hint:* Take a look at the `sklearn.naive_bayes import MultinomialNB` class. Type `help(sklearn.naive_bayes import MultinomialNB)` to see how to use its methods. Set `fit_prior=False`.

> *Hint:* To specify that the data is categorical, use the `sklearn.preprocessing.OneHotEncoder`. Otherwise `MultinomialNB` assumes that the input numbers are e.g. discrete counts of words or tokens. Without the encoding, the value 26 for a given element would signify 26 counts of a given tokens, whereas the encoding ensures that the value 26 is interpreted a level of a categorical variable corresponding to the letter *z*.

**Task 4.5:** Change to use the empirical prior instead. Does this improve the classifier performance and if so, why?

**Task 4.6:** Try classifying names based on only two of the letters in the name. You can do this by writing, e.g., `X=X[:,0:2]` to choose the first two letters. Are the first or last letters more useful for classifying names? Can you explain why?

In [ ]:
# Encode the categorical data using OneHotEncoder
X_encoded = OneHotEncoder().fit_transform(X=X[:, 0:4]) # using all 4 letters,

# Define the Naive Bayes classifier
fit_prior = True                                   # Set to False to use a uniform prior
classifier = MultinomialNB(fit_prior=fit_prior)
# Fit the classifier to the data
classifier.fit(X_encoded, y.ravel())

# Compute the posterior probabilities and name the variable y_est_prob and y_est
# YOUR CODE HERE
raise NotImplementedError()

# Get the misclassifications
misclassified = y_est != y.ravel()
print(f"Misclassification rate: {np.mean(misclassified):.2%}")

# Extract the first 100 misclassified names
names_misclassified = names[misclassified][:100]
print("\nFirst 100 misclassified names:")
print(names_misclassified)



<a id="my-anchor"></a>

---

# Assignment 4: 

In this assignment we use the **Whisky dataset** from weeks 2–3 and review marginal and conditional distributions, Bayes and Naïve Bayes.

In [ ]:
# Imports and setup for the assignment (same as week 3)
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')
sns.set_theme(font_scale=1.)

import utils as utils
utils.reset_marks()
a, b, c, d = utils.platform_info()
print(f"Platform info: <<<{a}:{b}:{c}:{d}>>>")
with utils.marks(0):
    assert True

<br>

**Assignment 4.0:** Fill in your student ID and your full name in the variables below (and ensure that the cell runs):

In [ ]:
student_id = "sXXXXXX"
student_typewritten_signature = "Firstname Lastname"

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
with utils.marks(0):
    assert re.compile(r"^s\d{6}$").match(student_id), "student_id should match pattern sXXXXXX"
    assert len(student_typewritten_signature) > 0, "signature must be non-empty"

Below we go through the dataloading and preparation so that you can focus on the probabilistic problems

1. We load the whisky data the same way as in weeks 2–3 from `data/whiskies.txt` and store the distillery names in `distilleries`.

2. Then, we extract the 12 flavour columns (drop RowID, Distillery, Postcode, ` Latitude`, ` Longitude`) and store the numeric matrix as `X`. Note: Here we opt to extract the matrix as a NumPy array, but you could, in princple, keep the Panda dataframe if you want.

3. We create a list `islay_whiskies` with Islay distillery names (e.g. Ardbeg, Bowmore, Bruichladdich, Bunnahabhain, Caol Ila, Lagavulin, Laphroaig). Set `y` to a boolean array: `True` if the distillery is in that list, else `False`.

4. We binarize the flavours: compute the median of each column of `X` (axis=0) and set `X_bin` to 1 where $ X \geq$ median and 0 otherwise.

In [ ]:
# Load the whisky dataset
df = pd.read_csv('data/whiskies.txt')
distilleries = df['Distillery']

# Extract the needed columns
X = df.drop(columns=['RowID', 'Distillery', 'Postcode', ' Latitude', ' Longitude']).values

# Get the whiskies from Islay and name the variable y
islay_whiskies = ['Ardbeg', 'Bowmore', 'Bruichladdich', 'Bunnahabhain', 'Caol Ila', 'Lagavulin', 'Laphroaig']
y = np.array([d in islay_whiskies for d in distilleries])

# Binanarize the features by comparing to the median and name the variable X_bin
medians = np.median(X, axis=0)
X_bin = (X >= medians).astype(int)

The **marginal** probability of an event is its probability without conditioning on other variables, obtained by summing or integrating over those variables.

**Assignment 4.1:** We set `N = len(y)` and you should compute the proportion of distilleries that are from Islay, i.e., compute $P(\text{Islay=True})$. Store in `P_islay_true`.

In [ ]:
n = len(y) # number of samples (should be 86)

# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Islay=True) = {P_islay_true:.3f}')

In [ ]:
with utils.marks(1):
    assert utils.check_scalar(P_islay_true, "0x5ac723f7")

**Assignment 4.2:** Compute the probability of not being from Islay and save in `P_islay_false` (we print both `P_islay` and `P_islay_false`; check that they sum to 1).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Islay=True) = {P_islay_true:.3f}')
print(f'P(Islay=False) = {P_islay_false:.3f}')

In [ ]:
with utils.marks(1):
    assert utils.check_scalar(P_islay_false, "0x1223bf19")

A **conditional probability** $P(A \mid B)$ is the probability of $A$ **given** that $B$ occurred. We use the **Smoky** flavour column.

We get the column index for Smoky with `flavour_cols.index('Smoky')` and create a boolean mask `smoky_high` (True where that column of `X_bin` is 1).

In [ ]:
flavour_cols = ['Body', 'Sweetness', 'Smoky', 'Medicinal', 'Tobacco', 'Honey', 'Spicy', 'Winey', 'Nutty', 'Malty', 'Fruity', 'Floral']
smoky_idx = flavour_cols.index('Smoky')
smoky_high = (X_bin[:, smoky_idx] == 1)

**Assignment 4.3:** Compute $P(\text{Smoky\_high})$ and $P(\text{Smoky\_low})$ (proportion of distilleries with that column 1 or 0). Store in `P_smoky_high` and `P_smoky_low`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Smoky = High) = {P_smoky_high:.3f}')
print(f'P(Smoky = Low) = {P_smoky_low:.3f}')

In [ ]:
with utils.marks(1):
    assert utils.check_scalar(P_smoky_high, "0x91713598")
    assert utils.check_scalar(P_smoky_low, "0xf609dc69")

**Assignment 4.4:** We create `islay_mask = (y == True)` to work out how many Whiskies from Islay there are and store as `n_islay`, then you should compute $P(\text{Smoky= High} \mid \text{Islay = True})$ = (number of times where Smoky is High and the Whisky is from Islay) / (number of Whiskies from Islay). Store in `P_smoky_high_given_islay_true`. 

In [ ]:
islay_mask = (y == True)
n_islay = islay_mask.sum()

# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Smoky=High | Islay=True) = {P_smoky_high_given_islay_true:.3f}')

In [ ]:
with utils.marks(1):
    assert utils.check_scalar(P_smoky_high_given_islay_true, "0xb44c37ea")

**Assignment 4.5:** Compute $P(\text{Smoky=High} \mid \text{Islay=False})$ = (number of times where Smoky is high and Islay is False) / (number of Islay being false). Store in `P_smoky_high_given_islay_false`. We then print all four probabilities.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Smoky=High) = {P_smoky_high:.3f},  P(Smoky_low) = {P_smoky_low:.3f}')
print(f'P(Smoky=High | Islay=True) = {P_smoky_high_given_islay_true:.3f},  P(Smoky=High | Islay=False) = {P_smoky_high_given_islay_false:.3f}')

In [ ]:
with utils.marks(1):
    assert utils.check_scalar(P_smoky_high_given_islay_false, "0x3792dfc4")

**Assignment 4.6:** We now have $P(\text{Smoky =High} \mid \text{Islay=True})$ and $P(\text{Smoky=High} \mid \text{Islay=False})$. Is this information alone to assess whether Smoky is a good predictor of whether a randomly chosen whisky comes from Islay? If not, explain what additional information would be required and why. 

- *Answer* (max 200 words):

[YOUR ANSWER HERE]

In [ ]:
# A manual test to be assessed by the teaching assistants (not auto-graded)
with utils.marks(1, auto=False, visible=False):
    print("Manual test")

**Bayes’ theorem**

We are now interested in using **Bayes’ theorem** to compute $\small P(\text{Islay=True} \mid \text{Smoky=High}) = \frac{P(\text{Smoky=High} \mid \text{Islay=True})\, P(\text{Islay=True})}{P(\text{Smoky=High})}$. The denominator comes from the **law of total probability**: $\small P(\text{Smoky=High}) = P(\text{Smoky=High} \mid \text{Islay=True}) P(\text{Islay=True}) + P(\text{Smoky=High} \mid \text{Islay=False}) P(\text{Islay=False})$.

**Assignment 4.7:** Compute $P(\text{Smoky=High})$ using the law of total probability above and store in `P_smoky_high_denom` (note we call it `P_smoky_high_denom` to illustrate that it is computed explicitly from law of total probability unlike `P_smoky_high`from Assignment 4.3, yet they should result in the same value just computed in different ways).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Smoky=High) = {P_smoky_high_denom:.3f} (calculated using the law of total probability)')

In [ ]:
with utils.marks(1):
    assert utils.check_scalar(P_smoky_high_denom, "0x91713598")

**Assignment 4.8:** Compute $\small P(\text{Islay=True} \mid \text{Smoky = High}) = \frac{P(\text{Smoky=High} \mid \text{Islay=True})\, P(\text{Islay=True})}{P(\text{Smoky=High})}$ and save as `P_islay_true_given_smoky_high`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Islay=True | Smoky=High) = {P_islay_true_given_smoky_high:.3f}')

In [ ]:
with utils.marks(1):
    assert utils.check_scalar(P_islay_true_given_smoky_high, "0x34da3ef2")

**Assignment 4.9:** We now consider two attributes, namely smoky and fruity. We wish to compute the probability of the Whisky **not** coming from Islay given that Smoky is high and Fruity is low.

Compute $\small P(\text{Islay=False} \mid \text{Smoky=High}, \text{Fruity=Low})$ using Bayes theorem and save as `P_islay_false_given_smoky_high_and_fruity_low`.  

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Islay = False | Smoky = High, Fruity = Low) = {P_islay_false_given_smoky_high_and_fruity_low:.3f}')

In [ ]:
with utils.marks(2):
    assert utils.check_scalar(P_islay_false_given_smoky_high_and_fruity_low, "0xfb556c8e")

**Assignment 4.10:** We now want to solve the task from Assignment 4.9 using the **Naïve Bayes** approach. 

Compute $P(\text{Islay=False} \mid \text{Smoky=High}, \text{Fruity=Low})$ using a **Naïve Bayes** classifier, and save the result as `P_islay_false_given_smoky_high_and_fruity_low_nb`.

> *Note:* This is intended to be a challenging task. We recommend writing out the equations on paper before starting the implementation.

In [ ]:
# Note: This will likely require several lines of code.

# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Islay = False | Smoky = High, Fruity = Low) ≈ {P_islay_false_given_smoky_high_and_fruity_low_nb:.3f} (computed using Naiive Bayes hence the ≈ symbol)' )

***Reflection (not assessed)*** Compare the Naiive Bayes result wiht the Bayes and determine if it gives the same result. Consider under which circumstances Bayes and Naiive Bayes gives exactly the same result.

In [ ]:
with utils.marks(2): 
    assert utils.check_scalar(P_islay_false_given_smoky_high_and_fruity_low_nb, "0xa613a37")

**Assignment 4.10**: Naïve Bayes with continuous feature conditionals. We now move from binarized features (e.g. *high* / *low*) to continuous features, which we model using Gaussian (Normal) distributions in the Naïve Bayes framework. Under the Naïve Bayes assumption, each feature is conditionally independent given the class label, and each **class-conditional feature distribution** is modeled as a univariate Normal distribution.For a continuous feature $ x_k $, we assume $\small 
p(x_k \mid \text{Islay}=c)
= \mathcal{N}\!\left(x_k \mid \mu_{k\mid c}, \sigma_{k,c}^2\right)$ here $\mu_{k\mid c}$ is the mean of feature $ k $ in class $ c $, $\sigma_{k \mid c}^2$  is the variance of feature $ k $ in class $ c $. Given a feature vector  the Naïve Bayes posterior is:
$$\small
P(y=c \mid \mathbf{x})
=
\frac{
P(y=c)\,\prod_{k=1}^{M}
\mathcal{N}\!\left(x_k \mid \mu_{k\mid c}, \sigma_{k\mid c}^2 \right)
}{
\sum\limits_{c'}
P(y=c')\,\prod_{k=1}^{M}
\mathcal{N}\!\left(x_k \mid \mu_{k\mid c'}, \sigma_{k\mid c'}^2 \right)
}
$$

where $ P(y=c) $ is the class prior, $p(\mathbf{x})$ is the evidence (normalization constant). The product arises from the **conditional independence assumption**.

We begin by defining the univariate Normal probability density function and implementing our own version. 
$
p(x \mid \mu, \sigma)
= \mathcal{N}(x \mid \mu, \sigma^2)
= \frac{1}{\sqrt{2\pi}\,\sigma}
\exp\!\left(
-\frac{(x-\mu)^2}{2\sigma^2}
\right)
$
This function will be used to evaluate each feature likelihood in the Naïve Bayes model.

In [ ]:
# Helper function: Our own Gaussian PDF (pure NumPy) to show that the Normal distrbution is just a function.
def my_normal_pdf(x, mu, var):    
    return (1.0 / np.sqrt(2.0 * np.pi * var)) * np.exp(-0.5 * ((x - mu) ** 2) / var)

**Assignment 4.10.1**  We include only two features in our model, Smoky and Fruity and want to compute $\small P(Islay=False \mid x_{Smoky}=2.87, x_{Fruity}=1.25)$. 

First we need to estimate the parameters of the class conditionals. As part of the we need the class conditionals, i.e. we need to estimate the parameter of the following distrbutions from the data in `X` (we use a very detailed/tendious notation to make it very explicit what we are looking for)

$$\small p(x_{Smoky} \mid Islay = True) = \mathcal{N}(x_{Smoky} \mid \mu_{Smoney | Islay=True} ,\sigma_{Smoky | Islay=True}^2) \quad \quad p(x_{Smoky} \mid Islay = False) = \mathcal{N}(x_{Smoky} \mid \mu_{Smoney | Islay=False} ,\sigma_{Smoky | Islay=False}^2)$$
$$\small  p(x_{Fruity} \mid Islay = True) = \mathcal{N}(x_{Fruity} \mid \mu_{Fruity | Islay=True} ,\sigma_{Fruity | Islay=True}^2) \quad \quad p(x_{Fruity} \mid Islay = False) = \mathcal{N}(x_{Fruity} \mid \mu_{Fruity | Islay=False} ,\sigma_{Fruity | Islay=False}^2)$$

E.g. the parameters of the first distrbution are $\mu_{Smoney | Islay=True}$ and $\small \sigma_{Smoky | Islay=True}^2$ the emperical mean and emperical (unbiased) variance.

In [ ]:
# As start we provide the solution for the first of the class conditionals, and 
# you need to complete the rest (which is very similar to the first one, 
# just with different masks and indices).
# The required naming can be deduced from the print statements at the end.

mu_smoky_given_islay_true   = X[islay_mask, smoky_idx].mean()
var_smoky_given_islay_true  = X[islay_mask, smoky_idx].var(ddof=1) # we set ddof=1 to get the unbiased variance estimate

# YOUR CODE HERE
raise NotImplementedError()

print("Class-conditional Gaussian models:\n")
print(f"p(x_Smoky | Islay = True )  =  N(μ = {mu_smoky_given_islay_true:.2f}, σ² = {var_smoky_given_islay_true:.2f})")
print(f"p(x_Smoky | Islay = False)  =  N(μ = {mu_smoky_given_islay_false:.2f}, σ² = {var_smoky_given_islay_false:.2f})")
print(f"p(x_Fruity | Islay = True)  =  N(μ = {mu_fruity_given_islay_true:.2f}, σ² = {var_fruity_given_islay_true:.2f})")
print(f"p(x_Fruity | Islay = False) =  N(μ = {mu_fruity_given_islay_false:.2f}, σ² = {var_fruity_given_islay_false:.2f})")

In [ ]:
# Helper function for visualizing the class-conditionals
xs_smoky = xs_fruity = np.linspace(0, 4, 400)

# Evaluate the Gaussian densities in the specified range
p_smoky_true  = my_normal_pdf(xs_smoky,  mu_smoky_given_islay_true,  var_smoky_given_islay_true)
p_smoky_false = my_normal_pdf(xs_smoky,  mu_smoky_given_islay_false, var_smoky_given_islay_false)
p_fruity_true  = my_normal_pdf(xs_fruity, mu_fruity_given_islay_true,  var_fruity_given_islay_true)
p_fruity_false = my_normal_pdf(xs_fruity, mu_fruity_given_islay_false, var_fruity_given_islay_false)

# Extract data by class
xsmoky_true   = X[ islay_mask, smoky_idx]
xsmoky_false  = X[~islay_mask, smoky_idx]
xfruity_true  = X[ islay_mask, fruity_idx]
xfruity_false = X[~islay_mask, fruity_idx]

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
plots = [(ax[0], xsmoky_true,  xsmoky_false,  xs_smoky,  p_smoky_true,  p_smoky_false,  "Smoky",  "x_Smoky"),
    (ax[1], xfruity_true, xfruity_false, xs_fruity, p_fruity_true, p_fruity_false, "Fruity", "x_Fruity")]
for a, x_t, x_f, xs, p_t, p_f, name, xlabel in plots:
    # shared bin edges for both classes
    bins = np.linspace(min(x_t.min(), x_f.min()), max(x_t.max(), x_f.max()),11)
    a.hist(x_t, bins=bins, density=True, alpha=0.4, label="Islay=True (hist)")
    a.hist(x_f, bins=bins, density=True, alpha=0.4, label="Islay=False (hist)")
    a.plot(xs, p_t, label="Islay=True (Normal density)")
    a.plot(xs, p_f, label="Islay=False (Normal density)")
    a.set(title=f"{name}", xlabel=xlabel, ylabel="density ")
    a.legend(frameon=False)

> **Reflection (no marks)**: Based on these visualizations, reflect on whether the Normal distribution is, ***strictly speaking***, an appropriate model for the (raw) data used here. In particular, consider which values may be generated when sampling from a Normal distribution and whether its density adequately captures the characteristics of the data, especially given the strong skew observed for *Smoky* when $Islay=True$. Regardless, we will continue to use the Normal distribution as an approximation.

In [ ]:
with utils.marks(0):
    assert utils.check_scalar(mu_smoky_given_islay_true, "0xec208e2b")
    assert utils.check_scalar(mu_smoky_given_islay_false, "0xf9bd4e3c")
    assert utils.check_scalar(var_smoky_given_islay_true, "0xc7564e60")
    assert utils.check_scalar(mu_smoky_given_islay_false, "0xf9bd4e3c")
    assert utils.check_scalar(var_smoky_given_islay_false, "0xd45cface")
    assert utils.check_scalar(mu_fruity_given_islay_true, "0xf3921e2f")
    assert utils.check_scalar(var_fruity_given_islay_true, "0xecf867f5")
    assert utils.check_scalar(mu_fruity_given_islay_false, "0x14713d2c")
    assert utils.check_scalar(var_fruity_given_islay_false, "0x9f185d72")

**Assignment 4.10.1** Put everything together and compute $\small P(Islay=False \mid x_{Smoky}=2.87, x_{Fruity}=1.25)$ (using Naiive Bayes). Save the result in `P_islay_false_given_smoky_val_and_fruity_val_nbc`.

> *Hint*: First evaluate the class-conditional likelihoods for the given inputs using `my_normal_pdf` and your estimated parameters. Then apply the Naive Bayes formula to compute the posterior probability.

In [ ]:
# Set the values we are interested in for the features (the ones we want to compute the posterior for)
x_smoky_val = 2.87
x_fruity_val = 1.25

# Likelihood terms: p(x_feature | class)
# We give p(x_k | y=c) for the first feature and class, you need to compute the three remaning ones (and then compute the posterior)
p_smoky_val_given_islay_true   = my_normal_pdf(x_smoky_val ,  mu_smoky_given_islay_true,  var_smoky_given_islay_true)

# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Islay=False | Smoky={x_smoky_val}, Fruity={x_fruity_val}) = {P_islay_false_given_smoky_val_and_fruity_val_nbc:.3f} (computed using Naiive Bayes)' )

In [ ]:
with utils.marks(3):
    assert utils.check_hash(P_islay_false_given_smoky_val_and_fruity_val_nbc, ((), 3.0887950606699066))

**Assignment 4.11: (optional challenge; no marks)** Ambitious students may wish to implement the Bayes version with continuous features (i.e., without relying on the Naive Bayes assumption):

$P(y=c \mid \mathbf{x})=\frac{P(y=c)\,\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_c, \mathbf{\Sigma}_c)}{\sum_{c'} P(y=c')\,\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_{c'}, \mathbf{\Sigma}_{c'})}$.

Here, $\boldsymbol{\mu}_c$ denotes the empirical mean vector of length $M$ for class $c$, and $\mathbf{\Sigma}_c$ denotes the $M \times M$ empirical (unbiased) covariance matrix for class $c$. Use the Bayes classifier to compute $\small P(Islay=False \mid x_{Smoky}=2.87, x_{Fruity}=1.25)$. Save the result in `P_islay_false_given_x_val`.

> *Note*: This question carries no marks and is intended for your own practice. It will likely require fewer lines of code than the Naive Bayes version. Moreover, this formulation can easily be reduced to the Naive Bayes case by considering only the diagonal elements of the covariance matrix.

In [ ]:
# Helper function. We define our own multivariate Gaussian PDF to show how the multivariate Normal distribution can be easily implemented as a function. 
# This is not meant to be an efficient implementation, but rather to show the mathematical form of the multivariate Normal distribution. 
# It uses some matrix tricks to ensure nummerical stability that you do not have to understand in this context
def my_mvn_pdf(x, mu, Sigma, eps=1e-9):
    x, mu = np.asarray(x), np.asarray(mu)
    d = x.size
    L = np.linalg.cholesky(Sigma + eps * np.eye(d))
    y = np.linalg.solve(L, x - mu)
    return np.exp(-0.5 * (d*np.log(2*np.pi) + 2*np.log(np.diag(L)).sum() + y@y))
    # Or without the Cholesky decomposition (but less numerically stable): return (1.0 / np.sqrt(np.power(2.0 * np.pi, d) * np.linalg.det(Sigma))) * np.exp(-0.5 * ((x - mu).T @ np.linalg.inv(Sigma) @ (x - mu)))

In [ ]:
P_islay_false_given_x_val = None # you need to override this value with the correct scalar value

# YOUR CODE HERE
raise NotImplementedError()

print(f'P(Islay=False | x) = {str(P_islay_false_given_x_val)}  (computed using Bayes)')

In [ ]:
with utils.marks(0, auto=True, visible=True):
    if P_islay_false_given_x_val is None:
        print("No attempt detected for the last question, hence no test to show.")
    else:
        assert utils.check_hash(P_islay_false_given_x_val,((), 2.5154129932938583))

<br>

**Assignment 4.12:** Declaration on the use of AI for solving this assignment (mandatory)

Answer by replacing *[your answer here]* with your response.
1. *I abide by [DTU's code of honour](https://student.dtu.dk/en/exam/exam-cheating/dtu-code-of-honour) and take full responsibility for the content of this submission (yes / no)*: 
    - [your answer here] 
2. *To what extent did you use generative AI to solve this assignment (0-100%)*:
    - [your answer here] 
3. *What was the primary use of generative AI, if any (writing / coding / other)*: 
    - [your answer here] 
4. *I feel I understand the key techniques/algorithms/methods/coding elements used in this exercise such that I can apply it in a **no aids** exam (yes/no)*: 
    - [your answer here] 

In [ ]:
with utils.marks(0, auto=False, visible=False):
    print("Manual test")

**Assignment 4.13:** To submit this notebook, make sure you have run everything in the **Assignment**-part, and convert this notebook to an HTML file.

> *Hint:* Open the **command palette** in Visual Studio Code, by pressing `Cmd + Shift + P` on Mac or `Ctrl + Shift + P` on Windows. 

> *Hint:* Search for `Jupyter: Export to HTML` and save the HTML file. 

> *Hint:* If you are running the notebook in the browser via the Jupyter interface, go to `File`, then `Save` and choose `Save and Export Notebook` as and select HTML.


**Assignment 4.14:** Hand in your `.ipynb` and `.HTML` file under assignments on DTU Learn for the relevant week.

**<h4>Summary of the points (only valid after rerunning the Assignment-part from scratch)</h4>**

In [ ]:
utils.marks_summary()
print("NOTE: Only valid after rerunning the assignment cells in the correct order")